# 05 - Performance Tuning & Operations

Single dataset: NYC Taxi.

Coverage:
- AQE: coalescing, runtime join switching, skew mitigation, and limits
- Broadcast variables and accumulators (including retry semantics)
- Dynamic resource allocation requirements and trade-offs
- Speculative execution for stragglers
- Fault tolerance: retries, lineage recomputation, DAG recovery, checkpointing


## Setup
All experiments are configuration-driven and timing-measured.


In [ ]:
from pathlib import Path
import os, time
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark import TaskContext


def _root():
    env = os.getenv("SPARK_TUNING_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Set SPARK_TUNING_PROJECT_ROOT")

ROOT = _root()
DATA = Path(os.getenv("SPARK_TUNING_DATA_DIR", str(ROOT / "data"))).resolve()
TAXI = DATA / "taxi"
TMP = DATA / "tmp"
CP = TMP / "checkpoints_05"
os.makedirs(TMP, exist_ok=True)
os.makedirs(CP, exist_ok=True)

print("ROOT", ROOT)
print("DATA", DATA)
print("TAXI", TAXI)


In [ ]:
taxi = spark.read.parquet(str(TAXI))
trips = taxi.select("VendorID","PULocationID","DOLocationID","payment_type","fare_amount","tip_amount","total_amount")
zone = taxi.groupBy("PULocationID","DOLocationID").agg(F.count("*").alias("trip_count"), F.avg("total_amount").alias("avg_total"))
loc = taxi.select("PULocationID","RatecodeID","Airport_fee").dropDuplicates(["PULocationID"])

print("app", spark.sparkContext.applicationId)
print("master", spark.sparkContext.master)
print("rows", taxi.count())
print("zone rows", zone.count())
print("loc rows", loc.count())


## Helpers


In [ ]:
def run_and_report(label, fn):
    sc = spark.sparkContext
    tr = sc.statusTracker()
    gid = f"nb05_{int(time.time()*1000)}"
    sc.setJobGroup(gid, label)
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    jobs = list(tr.getJobIdsForGroup(gid))
    stages = set()
    for j in jobs:
        ji = tr.getJobInfo(j)
        if ji is not None:
            stages.update(list(ji.stageIds))
    print(f"{label} -> {dt:.2f}s jobs={jobs} stages={sorted(stages)}")
    for s in sorted(stages):
        si = tr.getStageInfo(s)
        if si is not None:
            print(f"  stage {s}: tasks total={si.numTasks} done={si.numCompletedTasks} failed={si.numFailedTasks}")
    sc.setJobGroup("", "")
    return out, dt


def show_nodes(df, keys=("AdaptiveSparkPlan","CustomShuffleReader","SortMergeJoin","BroadcastHashJoin","Exchange","BatchEvalPython")):
    p = df._jdf.queryExecution().executedPlan().toString()
    for ln in p.splitlines():
        if any(k in ln for k in keys):
            print(ln.strip())


def set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20):
    spark.conf.set("spark.sql.adaptive.enabled", str(enabled).lower())
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", str(coalesce).lower())
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", str(skew).lower())
    spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", str(adaptive_broadcast_mb*1024*1024))


def exec_count():
    return len(spark.sparkContext.statusTracker().getExecutorInfos())


## Section 1 - Adaptive Query Execution (AQE)

AQE rewrites parts of the physical plan **after each shuffle stage completes**, using actual partition sizes and row counts rather than planner estimates. Enabled by default in Spark 3.0+.

Three runtime optimizations:
1. **Partition coalescing** - merges many small post-shuffle partitions into fewer larger ones (`CustomShuffleReader` in plan)
2. **Join strategy switching** - if a join side shrinks below the broadcast threshold after filtering, switches SortMergeJoin -> BroadcastHashJoin at runtime without restarting the query
3. **Skew join mitigation** - detects oversized shuffle partitions, splits them into sub-tasks

Key configs:
- `spark.sql.adaptive.enabled` - master switch (default: true in Spark 3+)
- `spark.sql.adaptive.coalescePartitions.enabled`
- `spark.sql.adaptive.skewJoin.enabled`
- `spark.sql.adaptive.autoBroadcastJoinThreshold` - runtime threshold, separate from static planner

**How to read AQE plans**: before any action the plan shows `AdaptiveSparkPlan isFinalPlan=false` - a prediction. After an action: `isFinalPlan=true` with actual `CustomShuffleReader` nodes. Always call `show_nodes()` **after** `run_and_report()` for AQE demos.

**What AQE does NOT fix**: Python UDF serialization overhead, incorrect query logic, plans with no shuffle.

### Demo A - partition coalescing

Set 400 shuffle partitions on a dataset that produces far fewer distinct groups. With AQE OFF: 400 tiny tasks. With AQE ON: `CustomShuffleReader` merges them. Check Spark UI -> Stages: compare task count.

In [7]:
spark.conf.set("spark.sql.shuffle.partitions", "400")

set_aqe(enabled=False)
q_off = trips.groupBy("PULocationID", "DOLocationID").agg(F.count("*").alias("c"), F.avg("total_amount").alias("a"))
print("AQE OFF - exchange nodes:")
show_nodes(q_off)
_, t_off = run_and_report("coalescing AQE OFF", lambda: q_off.count())

set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20)
q_on = trips.groupBy("PULocationID", "DOLocationID").agg(F.count("*").alias("c"), F.avg("total_amount").alias("a"))
_, t_on = run_and_report("coalescing AQE ON", lambda: q_on.count())

# Note: executedPlan() on a DataFrame after count() returns a fresh unevaluated plan (isFinalPlan=false).
# The real evidence of coalescing is in the stage task counts above: AQE OFF = 400 tasks, AQE ON = much fewer.
print(f"\nAQE off: {t_off:.2f}s  on: {t_on:.2f}s")

AQE OFF - exchange nodes:
+- Exchange hashpartitioning(PULocationID#7, DOLocationID#8, 400), ENSURE_REQUIREMENTS, [plan_id=856]
coalescing AQE OFF -> 0.51s jobs=[18] stages=[39, 40, 41]
  stage 39: tasks total=12 done=12 failed=0
  stage 40: tasks total=400 done=400 failed=0
  stage 41: tasks total=1 done=1 failed=0
coalescing AQE ON -> 0.22s jobs=[21, 20, 19] stages=[42, 43, 44, 45, 46, 47]
  stage 42: tasks total=12 done=12 failed=0
  stage 43: tasks total=12 done=0 failed=0
  stage 44: tasks total=4 done=4 failed=0
  stage 45: tasks total=12 done=0 failed=0
  stage 46: tasks total=4 done=0 failed=0
  stage 47: tasks total=1 done=1 failed=0

AQE off: 0.51s  on: 0.22s


### Demo B - runtime join strategy switching

Disable static broadcast so the planner picks SortMergeJoin. After the first shuffle stage,
AQE measures the actual size of the small side - if it fits under the adaptive threshold (32 MB),
it switches SortMergeJoin -> BroadcastHashJoin at runtime without restarting the query.

Right side: `loc` - 263 unique locations (tiny). Static planner can't broadcast it (threshold=-1),
but AQE can after measuring the actual shuffle output.

Check Spark UI -> SQL -> click the query: look for BroadcastHashJoin instead of SortMergeJoin.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=32)

# loc has 263 rows - tiny. Static planner won't broadcast (threshold=-1), so it plans SMJ.
# After the first shuffle AQE measures loc's actual size and switches to BHJ.
switch_q = trips.join(loc, on="PULocationID", how="inner").groupBy("PULocationID").agg(F.count("*").alias("c"))

print("Pre-execution plan (SortMergeJoin - broadcast disabled statically):")
show_nodes(switch_q)
_, t_sw = run_and_report("runtime join switching", lambda: switch_q.count())
# Post-execution AQE plan is only visible in Spark UI -> SQL -> click the query.
# Look for BroadcastHashJoin - AQE should have switched from SMJ after measuring loc's shuffle output.
print(f"\nseconds: {t_sw:.2f}")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

### Demo C - skew join mitigation

Partition skew as a join problem is covered in depth in notebook 03. AQE skew mitigation builds on top of that: after the shuffle map stage completes, AQE reads the actual partition sizes and splits any partition that exceeds `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (default 256 MB) AND is more than `skewedPartitionFactor` (default 5×) larger than the median.

What this looks like in Spark UI when it triggers:
- **SQL plan**: `SortMergeJoin` node shows `isSkewJoin=true`
- **Stage Tasks**: the join stage has more tasks than `shuffle.partitions` - the skewed partition was split into sub-tasks
- **Stages**: the straggler task duration drops compared to the OFF run

**Requirement**: skew mitigation only works on SortMergeJoin - both sides must go through a ShuffleQueryStage. If either side gets broadcast (via static or adaptive threshold), AQE has nothing to split.

## Section 2 - Broadcast Variables and Accumulators

**Broadcast variables** distribute a read-only value from the driver to each executor **once**. Without broadcast, a task closure sends the object with every task - for a 100-task job that's 100 copies over the network. With broadcast: 2 executors × 1 copy = 2 copies total.

Lifecycle:
- `sc.broadcast(value)` - distribute to executors (sent lazily on first `.value` access)
- `.value` inside a task - reads the executor-local cached copy
- `.unpersist()` - evict from executor memory (re-sent on next access)
- `.destroy()` - permanent destruction; any subsequent `.value` raises an exception

**When to use**: lookup tables, ML models, config dicts in RDD operations or Python UDFs. DataFrame joins use broadcast automatically via `autoBroadcastJoinThreshold` - no manual broadcast needed there.

**Accumulators** are driver-readable counters that tasks can only add to. The write-only constraint in tasks means no synchronization needed.

**Retry semantics**: if a task is retried (failure or speculation), its accumulator increments run again - Spark does not deduplicate. Use accumulators for monitoring, not for counts that drive business logic.

### Demo A - broadcast variable

Broadcast a set of hot zone IDs to executors. Each executor receives it once; all tasks on that executor share the same in-memory copy.

In [ ]:
sc = spark.sparkContext
hot_zones = sc.broadcast({132, 138, 170, 237, 236, 161})

def mark_hot(rows):
    h = hot_zones.value  # executor-local read - one copy per executor, not per task
    for r in rows:
        yield (r.PULocationID in h, 1)

flags = trips.select("PULocationID").rdd.mapPartitions(mark_hot).toDF(["is_hot", "one"])
_, t = run_and_report("broadcast variable", lambda: flags.groupBy("is_hot").agg(F.sum("one").alias("cnt")).collect())
print(f"seconds: {t:.2f}")
hot_zones.unpersist()
hot_zones.destroy()

### Demo B - accumulator

Basic row counter. The accumulator increments once per row across all executors; the driver reads the total after the job completes.

**Retry double-counting**: if a task is retried (failure or speculation), its increments run again - Spark does not deduplicate. In this environment `spark.task.maxFailures` is static and cannot be changed at runtime, so the live retry demo is skipped. The principle: use accumulators for monitoring only, never for counts that drive business logic.

In [29]:
from pyspark import TaskContext

acc = sc.accumulator(0)
_, t = run_and_report("acc basic", lambda: trips.select("total_amount").rdd.map(lambda r: acc.add(1) or r).count())
print(f"rows counted: {acc.value:,}  seconds: {t:.2f}")

[Stage 247:==========================================>             (9 + 3) / 12]

acc basic -> 8.49s jobs=[118] stages=[247]
  stage 247: tasks total=12 done=12 failed=0
rows counted: 41,169,720  seconds: 8.49


## Section 3 - Dynamic Resource Allocation

DRA lets Spark request and release executors from the cluster manager at runtime. Idle executors are released after `spark.dynamicAllocation.executorIdleTimeout` (default 60s); new executors are requested when tasks queue up.

Key configs: `spark.dynamicAllocation.enabled`, `min/initial/maxExecutors`, `spark.shuffle.service.enabled`.

**The shuffle service requirement**: when an executor is released, its shuffle output files would be lost without an external shuffle service. The external shuffle service is a long-running sidecar process on each worker node (managed by YARN/Kubernetes in production). In local Standalone mode (this lab), the external shuffle service is not running - enabling DRA via config has no effect on executor count.

**When DRA helps**: long-lived interactive sessions, ETL pipelines with variable load phases, shared clusters.
**When DRA hurts**: very short jobs (executor startup latency dominates), pipelines with back-to-back shuffle stages.

### When DRA matters - example

**Shared cluster, morning ETL pipeline**:

1. Stage 1 - ingest and join 500 GB raw logs. DRA scales up to 20 executors.
2. Stage 2 - write a small summary table (5 partitions). 18 executors sit idle.
3. Without DRA: those 18 executors are locked for the rest of the job, blocking other teams' jobs.
4. With DRA: after Stage 1 completes, 18 executors are released back to the cluster within `executorIdleTimeout` (default 60s). Other jobs can use them immediately.

In this lab (local Standalone, no external shuffle service) DRA cannot release executors because shuffle files would be lost. The config exists and can be set, but executor count does not change.

## Section 4 - Speculative Execution

A **straggler** is a task that runs significantly longer than the median task in the same stage - typically due to hardware heterogeneity, data skew, GC pressure, or network issues.

With `spark.speculation=true`, Spark monitors task durations. When a task's duration exceeds `multiplier × median` (default: 1.5×) and at least `quantile` fraction of tasks in the stage are complete (default: 75%), Spark launches a duplicate attempt on a different executor. Whichever finishes first wins; the other is killed.

Key configs: `spark.speculation`, `spark.speculation.multiplier` (1.5), `spark.speculation.quantile` (0.75).

**Cost**: duplicated work. Only enable when you have spare executor capacity.
**Limitation**: speculation does not help if the straggler has genuinely more data - the duplicate gets the same large partition.
**Risk**: non-idempotent tasks (DB writes, external API calls) must not be speculated - the side effect runs twice.

### Speculative execution in practice

When it triggers, Spark UI -> Stages -> Tasks shows two rows for the same partition index - the original and the speculative attempt. Whichever finishes first is marked SUCCESS; the other is KILLED.

**Spark 4.0 note**: `spark.speculation` is now a static config and cannot be toggled at runtime. To enable it, set it in `spark-defaults.conf` or pass `--conf spark.speculation=true` when starting the session.

**Use case**: long-running batch jobs on heterogeneous hardware where occasional slow nodes cause tail latency. Not useful for jobs where all tasks are equally fast or where the cluster is already at full capacity.

## Section 5 - Fault Tolerance: Retries, Lineage, and Checkpointing

Spark's fault tolerance is **lineage-based**: the DAG of transformations is enough to recompute any lost partition from its sources. No checkpointing needed by default - just rerun the failed stage.

**Task-level retry**: `spark.task.maxFailures` (default 4). A failed task is retried on any available executor. Each attempt gets a new `attemptNumber()` from `TaskContext`. Retries are transparent to the caller - `df.count()` returns the correct result even if tasks were retried.

**Stage-level DAG recovery**: if an executor is lost after writing shuffle output, those shuffle files are gone. Spark recomputes the shuffle map stage from its input. The external shuffle service (Section 3) decouples shuffle file lifetime from executor lifetime to avoid this recompute.

**Note**: both mechanisms cannot be demonstrated live in this environment. `spark.task.maxFailures` is a static config in Spark 4.0 and is set to 1 here - injected failures abort immediately instead of retrying. Stage recovery requires killing a real executor, which is not possible on a single-node Docker cluster.

**RDD checkpointing** truncates lineage: Spark materializes the RDD to disk and forgets the computation chain. Useful for iterative algorithms with long lineage chains (ML training loops, graph algorithms) where full recompute on failure would be expensive.

- `sc.setCheckpointDir(path)` - must be reliable storage (HDFS, S3, local)
- `rdd.checkpoint()` - mark; materialized on the **next action**
- `rdd.isCheckpointed()` / `rdd.getCheckpointFile()` - verify

**Checkpoint vs cache**: `cache()` = fast, volatile (lost on executor death). `checkpoint()` = slower write, durable. Use both together in iterative jobs: cache for this iteration's speed, checkpoint every N iterations to truncate lineage.

### Demo A - lineage and checkpointing

Build a 4-step RDD chain and inspect the lineage string before and after checkpointing. After `checkpoint()` + one action, the lineage is truncated - the RDD now reads from the checkpoint file.

In [33]:
sc.setCheckpointDir(str(CP))
base = sc.parallelize(range(5_000_000), 24)
chained = base.map(lambda x: x+1).map(lambda x: x*2).filter(lambda x: x%3!=0).map(lambda x: x-7)

def lineage(rdd):
    d = rdd.toDebugString()
    s = d.decode("utf-8","ignore") if isinstance(d,bytes) else d
    return s[:500]

print("Lineage BEFORE:")
print(lineage(chained))
_, t_before = run_and_report("before checkpoint", lambda: chained.count())

chained.checkpoint()
_, t_after = run_and_report("checkpoint materialize", lambda: chained.count())

print("\nLineage AFTER (truncated):")
print(lineage(chained))
print(f"isCheckpointed: {chained.isCheckpointed()}  file: {chained.getCheckpointFile()}")
print(f"\nbefore: {t_before:.2f}s  after: {t_after:.2f}s")

Lineage BEFORE:
(24) PythonRDD[525] at RDD at PythonRDD.scala:56 []
 |   ParallelCollectionRDD[524] at readRDDFromFile at PythonRDD.scala:297 []
before checkpoint -> 0.41s jobs=[123] stages=[252]
  stage 252: tasks total=24 done=24 failed=0
checkpoint materialize -> 1.06s jobs=[125, 124] stages=[253, 254]
  stage 253: tasks total=24 done=24 failed=0
  stage 254: tasks total=24 done=24 failed=0

Lineage AFTER (truncated):
(24) PythonRDD[525] at RDD at PythonRDD.scala:56 []
 |   ReliableCheckpointRDD[528] at count at /tmp/ipykernel_705/1009512199.py:15 []
isCheckpointed: True  file: file:/data/tmp/checkpoints_05/9ca2e334-a8e7-4674-99a9-203e08ab4fca/rdd-525

before: 0.41s  after: 1.06s


## Operations runbook

**Tail latency**: check Spark UI -> Tasks for outlier by duration/input size. If data skew -> salt or AQE skew join. If hardware straggler -> speculation.
**Throughput collapse**: check shuffle volume, join strategy, `BatchEvalPython` in hot path.
**Repeated failures**: Executors tab for lost executors; fix OOM before raising memory limits.
**Long recomputation**: cache shared DataFrames; checkpoint iterative pipelines every N iterations.

## Key takeaways
- AQE acts at shuffle boundaries - understand static plans first (notebooks 01-03)
- Broadcast and accumulators need explicit lifecycle management and are retry-unsafe
- DRA requires external shuffle service - runtime toggle has no effect in local Standalone
- Speculative execution duplicates work - only enable with spare capacity
- Fault tolerance is lineage-first; checkpointing is an explicit optimization